# Phase 3A: zero-shot cross-lingual transfer

XLM-R Base fine-tuned on English only, evaluated on the same test questions in all five languages.

All numbers come from the exported run artifacts in `results/`. The current cohort is the provisional `gemini-v2` `span_checked` release (322 test question groups, human review pending), so results must be labelled provisional.

In [ ]:
import csv, json
from pathlib import Path
import matplotlib.pyplot as plt

RESULTS = Path.cwd().parent / "results" if Path.cwd().name == "notebooks" else Path("phase_03_multilingual_finetuning/results")
LANGS = ["en", "hi", "mr", "bn", "ta"]

def load_json(name):
    return json.loads((RESULTS / name).read_text(encoding="utf-8"))

def load_csv(name):
    with (RESULTS / name).open(encoding="utf-8", newline="") as stream:
        return list(csv.DictReader(stream))

def show(rows, fields=None, digits=2):
    """Print a plain-text table without extra dependencies."""
    fields = fields or list(rows[0])
    def fmt(value):
        try:
            return f"{float(value):.{digits}f}" if "." in str(value) else str(value)
        except ValueError:
            return str(value)
    cells = [[fmt(row.get(f, "")) for f in fields] for row in rows]
    widths = [max(len(f), *(len(c[i]) for c in cells)) for i, f in enumerate(fields)]
    print("  ".join(f.ljust(w) for f, w in zip(fields, widths)))
    for c in cells:
        print("  ".join(v.ljust(w) for v, w in zip(c, widths)))

def require(*names):
    missing = [n for n in names if not (RESULTS / n).exists()]
    if missing:
        raise FileNotFoundError("Run scripts/export_results.py first. Missing: " + ", ".join(missing))

In [ ]:
require("phase_3a/metrics.json", "phase_3a/predictions.jsonl")
metrics = load_json("phase_3a/metrics.json")
show(load_csv("phase_3a/language_wise_results.csv"))
print("English SQuAD-v1 score (separate protocol):", metrics["english_squad_v1"])

In [ ]:
scores = metrics["by_language_and_macro"]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([l.upper() for l in LANGS], [scores[l]["f1"] for l in LANGS], color="#416e9a", label="F1")
ax.bar([l.upper() for l in LANGS], [scores[l]["exact_match"] for l in LANGS], color="#9dbad6", label="EM", width=0.5)
ax.set_ylim(0, 100); ax.set_ylabel("%"); ax.legend(); ax.set_title("Phase 3A by language")
plt.show()

## English-to-Indic transfer loss

In [ ]:
show([{"language": l, "f1_gap_vs_english_pp": scores["en"]["f1"] - scores[l]["f1"]} for l in LANGS[1:]])

## Example errors (first five per Indic language)

In [ ]:
predictions = [json.loads(line) for line in (RESULTS / "phase_3a/predictions.jsonl").open(encoding="utf-8")]
for lang in LANGS[1:]:
    wrong = [p for p in predictions if p["language"] == lang and not p["exact_match"]][:5]
    print(f"--- {lang}")
    for p in wrong:
        print(f"{p['id']}: gold={p['gold_answers'][0]!r} predicted={p['predicted_answer']!r} f1={p['f1']:.2f}")